this is the first dry run with artificial 2d data and a suuuper simple network. just to get a feeling for the torch-parameter-manipulation-api and setting up the workflow

#### Setting up the network and data

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import numpy as np
from tqdm.auto import tqdm
import h5py
import plotly.graph_objects as go
from torch.func import functional_call
import matplotlib.pyplot as plt

from loss_landscapes.paths import PROJ_ROOT
from loss_landscapes.utils import plotly_template
plotly_template.use(name="mine_dark")

# little prototype network
class FuncNet(nn.Module):
    def __init__(self, dim_in: int = 2, dim_hidden: int = 128, dim_out: int = 1):
        super().__init__()
        self.fc1 = nn.Linear(dim_in, dim_hidden)
        self.fc2 = nn.Linear(dim_hidden, dim_hidden)
        self.fc3 = nn.Linear(dim_hidden, dim_out)
        self.act = nn.ReLU()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)
        return x
    
    
# Device setup 
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"⯈ using {DEVICE} as device")

# load dataset
print("⯈ loading dataset ...")
with h5py.File(PROJ_ROOT / "data/ds_2dfunc.h5", "r") as f:
    X = f["data"][...]
    y = f["targets"][...]
    
# define some vars
n_epochs = 10
batch_size = 128
train_perc = 0.8
n_samples = len(X)
n_train = int(n_samples * train_perc)
n_valid = n_samples - n_train

# setup training elements
model = FuncNet().to(DEVICE)
optim = torch.optim.Adam(params=model.parameters(), lr=1e-3)
X = torch.Tensor(X).to(DEVICE)
y = torch.Tensor(y)[:, None].to(DEVICE) # add singleton dim to y to match the model output dims

#### Training the test Network

In [ ]:
# run the prototypical loop & train the network

pbar = tqdm(range(n_epochs), desc="training: ")
for E in pbar:
    
    # training part -----
    train_idx = torch.randperm(n_train, device=DEVICE)
    model.train()
    losses_train = []
    
    for batch_start in range(0, n_train, batch_size): # start index of every batch, > step size batch_size
        batch_idx = train_idx[batch_start:batch_start+batch_size]

        optim.zero_grad()
        pred = model(X[batch_idx])
        loss = torch.mean((pred - y[batch_idx])**2)
        loss.backward()
        optim.step()
        
        losses_train.append(loss.detach())
      
        
    # validation part -----
    valid_idx = torch.arange(n_valid, device=DEVICE) + n_train # offset by n train to not use train samples!
    losses_valid = []
    model.eval()
    
    with torch.no_grad():
        for batch_start in range(0, n_valid, 1_000):
            batch_idx = valid_idx[batch_start:batch_start+1_000]
            
            pred = model(X[batch_idx])
            loss = torch.mean((pred - y[batch_idx])**2)
            
            losses_valid.append(loss.detach())
    
    
    # log the losses
    tqdm.write(
        f"L_train = {torch.stack(losses_train).mean().sqrt().item():.4f} | "
        f"L_valid = {torch.stack(losses_valid).mean().sqrt().item():.4f}"
    )
    


#### basic api for model params

In [ ]:
# # grab all the model params (but just params)
# params = list(model.parameters())

# # get a parameter count for example
# n = sum(p.numel() for p in model.parameters())

# # the state dict contains all the params + buffers (batch norm statistics). these are refs tho!
# sd = model.state_dict()

# # writing values into model params: load from state dict (names must match)
# model.load_state_dict(sd)

# with torch.no_grad():
#     model.fc1.weight[0, 0] = 5
    
# print(model.fc1.weight[0])

#### seeing how the parameters are distributed overall

In [ ]:
allparams = []

# collect aaaaalll model parameters on one list
for p in model.parameters():
    allparams += list(p.detach().to("cpu").numpy().ravel())
    
_ = plt.hist(allparams, bins=100)

#### first naive approach at evaluating 2 random parameter-space-directions

In [ ]:
def loss_at(params: dict, X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """ to evaluate a model with data AND parameters """
    pred = functional_call(model, params, (X, ))
    return torch.sqrt(torch.mean((pred - y) ** 2))

# two directions in parameter space + the trained weights fixed point
theta = {k: v.detach().clone() for k, v in model.named_parameters()}
d1 = {k: torch.randn_like(v) for k, v in model.named_parameters()}
d2 = {k: torch.randn_like(v) for k, v in model.named_parameters()}


reach = 0.1

alphas = np.linspace(-reach, reach, 20)
betas  = np.linspace(-reach, reach, 20)

evalX = X[:1000]
evaly = y[:1000]
Z = torch.zeros(len(betas), len(alphas))

with torch.no_grad():
    for j, b in enumerate(betas):
        for i, a in enumerate(alphas):
            params = {k: theta[k] + a*d1[k] + b*d2[k] for k in theta}
            Z[i, j] = loss_at(params, evalX, evaly)

    
fig = go.Figure()
fig.add_trace(
    go.Surface(
        x = alphas,
        y = betas,
        z = np.log(Z.numpy()),
    )
)

fig.show()

#### now with per filter normalization > the main idea of that paper as filters (the unit behind one ReLU) can scale independently of each other (filter: for convnets its conv kernels, for MLPs its just 1 neuron)

In [ ]:
# a bit on normalization

randn_vec = np.random.randn(1000)
vec_norm = np.sqrt(np.sum(randn_vec**2))

print(f"sqrt(1000) = {np.sqrt(1000):.2f} and norm of randn vec length 1000 = {vec_norm:.2f}")

# so if the direction "vector" should ultimately have the same norm as the original param filter vector
L = 100
unit_randn_vec = np.random.randn(L)/np.sqrt(L)
print(f"vector norm = {np.sqrt((unit_randn_vec**2).sum())}")

# but to be precise we divide by the actual norm
L = 100
randn_vec = np.random.randn(L)
vec_norm = np.sqrt(np.sum(randn_vec**2))
unit_randn_vec = randn_vec/vec_norm
print(f"vector norm = {np.sqrt((unit_randn_vec**2).sum())}")



In [ ]:
# so now normalized dirs

def loss_at(params: dict, X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """ to evaluate a model with data AND parameters """
    pred = functional_call(model, params, (X, ))
    return torch.sqrt(torch.mean((pred - y) ** 2))

def rownorm(tens: torch.Tensor):
    """ mlp layers are dim(out, in) so the rows are the filters > norm per row"""
    return torch.sqrt(torch.sum(tens**2, dim=1, keepdim=True))
    
def generate_dir(param_snapshot) -> dict:
    dirdict = {}
    
    for k, p in param_snapshot.items():
        
        if p.dim() > 1:
        
            randn_dir = torch.randn_like(p)
            normed_dir = randn_dir / rownorm(randn_dir)
            scaled_dir = normed_dir * rownorm(p)
            dirdict[k] = scaled_dir
        
        else: 
            dirdict[k] = torch.zeros_like(p)

    return dirdict
            

# two directions in parameter space + the trained weights fixed point
theta = {k: v.detach().clone() for k, v in model.named_parameters()}

d1 = generate_dir(theta)
d2 = generate_dir(theta)


reach = 1
N = 51

alphas = np.linspace(-reach, reach, N)
betas  = np.linspace(-reach, reach, N)

evalX = X[:5000]
evaly = y[:5000]
Z = torch.zeros(len(betas), len(alphas))

with torch.no_grad():
    for j, b in enumerate(betas):
        for i, a in enumerate(alphas):
            params = {k: theta[k] + a*d1[k] + b*d2[k] for k in theta}
            Z[j, i] = loss_at(params, evalX, evaly)
            
fig = go.Figure()
fig.add_trace(
    go.Surface(
        x = alphas,
        y = betas,
        z = np.log(Z.numpy()),
    )
)

fig.show()